# Physical-boundary jet RG evolution

This notebook solves the modified time-like DGLAP evolution on a $1000\times1000$ lattice with VFNS and plots the evolved quark and gluon jet functions at six scales.

The boundary lies at
$$
\mu_*(b)=\frac{b_0}{b_*(b)},\qquad
b_*(b)=\frac{b}{\sqrt{1+b^2/b_0^2}}.
$$
At $(b_*,\mu_*)$, the perturbative logarithm $L_b=\ln(\mu_*^2b_*^2/b_0^2)$ vanishes. The NNLO boundary is therefore
$$
J_i(b,\mu_*)=\left[1+a_s(\mu_*)J_i^{(1)}(n_f)+a_s^2(\mu_*)J_i^{(2)}(n_f)\right]F_i^{\rm NP}(b),
\qquad a_s=\frac{\alpha_s}{4\pi},
$$
with $F_q^{\rm NP}=e^{-2.3b}$ and $F_g^{\rm NP}=e^{-3.8b}$. Both $\alpha_s(\mu_*)$ and every coefficient containing $n_f$ use $n_f=n_f(\mu_*)$ from VFNS; no fixed-$n_f=5$ constants are used. The zero-log perturbative constants are the EEC jet-function coefficients of [Dixon, Moult, and Zhu](https://arxiv.org/abs/1905.01310), also reproduced in the local Mathematica notebook `Collinear EEC/jet NNLO.nb`.

In [ ]:
import os
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

os.environ.setdefault("JULIA_NUM_THREADS", "auto")

if "Main" not in globals():
    from julia.api import Julia

    julia_env = os.environ.get("JULIA_EXE")
    if julia_env:
        julia_executable = Path(julia_env)
    else:
        julia_candidates = sorted(
            (Path(os.environ["LOCALAPPDATA"]) / "Programs").glob("Julia-*/bin/julia.exe")
        )
        if not julia_candidates:
            raise FileNotFoundError("Set JULIA_EXE to the installed julia executable.")
        julia_executable = julia_candidates[-1]

    jl = Julia(runtime=str(julia_executable), compiled_modules=False, threads="auto")
    from julia import Main

print(f"Julia {Main.eval('string(VERSION)')} with {Main.eval('Threads.nthreads()')} threads")

In [ ]:
cwd = Path.cwd()
rg_dir = cwd if (cwd / "numerical_rg.jl").exists() else cwd / "Numerical RG"
Main.include(str(rg_dir / "numerical_rg.jl"))

n_nodes = 1000
b_min = 0.001       # Required to include mu = 500 GeV in the solved plane.
b_max = 50.0        # Zero-closure endpoint.
mu_values = np.array([10.0, 20.0, 50.0, 100.0, 200.0, 500.0])
b_plot = np.geomspace(0.01, 5.0, 500)

Main.n_nodes = n_nodes
Main.b_min = b_min
Main.b_max = b_max
Main.eval("bstar_func(b) = b / sqrt(1.0 + b^2 / b0^2)")
Main.eval("lattice = build_lattice_grid(n_nodes=n_nodes, b_min=b_min, b_max=b_max, bstar_func=bstar_func)")

mu_limits = np.exp(0.5 * np.asarray(Main.eval("[first(lattice.t_grid), last(lattice.t_grid)]"), dtype=float))
print(f"Solved mu range: {mu_limits[0]:.6g} to {mu_limits[1]:.6g} GeV")
assert mu_values.min() >= mu_limits[0] and mu_values.max() <= mu_limits[1]

In [ ]:
Main.eval(r"""
function jet_boundary_coefficients(nf::Int)
    zeta2 = pi^2 / 6.0
    zeta3 = 1.2020569031595942854
    zeta4 = pi^4 / 90.0

    jq1 = -(37.0 / 3.0) * CF
    jg1 = -(898.0 / 75.0) * CA - (14.0 / 25.0) * nf

    jq2 = (
        (152.0 * zeta4 - 478.0 * zeta3 - 106.0 * zeta2 + 3498505.0 / 5184.0) * CF^2 +
        (-76.0 * zeta4 + 280.0 * zeta3 + (1063.0 / 15.0) * zeta2 -
         164883727.0 / 324000.0) * CA * CF +
        ((9.0 / 5.0) * zeta2 + 703847.0 / 24000.0) * CF * nf
    )

    jg2 = (
        (76.0 * zeta4 - (1054.0 / 5.0) * zeta3 - (2159.0 / 75.0) * zeta2 +
         133639871.0 / 810000.0) * CA^2 +
        ((44.0 / 5.0) * zeta3 - (127.0 / 25.0) * zeta2 +
         68111303.0 / 1620000.0) * CA * nf +
        (4.0 * zeta3 + (14.0 / 5.0) * zeta2 - 1528667.0 / 108000.0) * CF * nf +
        (-(8.0 / 15.0) * zeta2 + 2344.0 / 1125.0) * nf^2
    )

    return (jq1=jq1, jq2=jq2, jg1=jg1, jg2=jg2)
end

nf_scheme = :VFNS
alpha_s_order = 4

# One numerical coupling evolution supplies every boundary node.
boundary_alpha_s_vec = alpha_s_grid(
    lattice.t_grid;
    order=alpha_s_order,
    nf_scheme=nf_scheme,
)
boundary_alpha_s_at_mu = Dict{Float64, Float64}(
    zip(lattice.mu_i_grid, boundary_alpha_s_vec)
)

function perturbative_boundary(mu_start::Float64)
    nf = nf_func(mu_start; scheme=nf_scheme)
    alpha_s = get(boundary_alpha_s_at_mu, mu_start) do
        alpha_s_func_numerical(
            mu_f=mu_start,
            order=alpha_s_order,
            nf_scheme=nf_scheme,
        )
    end
    a_s = alpha_s / (4.0 * pi)
    coefficient = jet_boundary_coefficients(nf)

    jq = 1.0 + a_s * coefficient.jq1 + a_s^2 * coefficient.jq2
    jg = 1.0 + a_s * coefficient.jg1 + a_s^2 * coefficient.jg2
    return (jq=jq, jg=jg, nf=nf, alpha_s=alpha_s)
end

function boundary_func(; b, bstar, mu_start)
    perturbative = perturbative_boundary(mu_start)
    return (
        perturbative.jq * exp(-2.3 * b),
        perturbative.jg * exp(-3.8 * b),
    )
end
""")

nf_on_boundary = np.asarray(
    Main.eval("[nf_func(mu; scheme=:VFNS) for mu in lattice.mu_i_grid]"),
    dtype=int,
)
print("Active flavors represented on the boundary:", np.unique(nf_on_boundary))

In [ ]:
start = time.perf_counter()
Main.eval(r"""
solution = solve_stepwise_rg(
    lattice=lattice,
    boundary_func=boundary_func,
    order=2,
    nf_scheme=:VFNS,
    alpha_s_order=4,
    method=:rk2,
    closure_check=:error,
)
""")
solve_seconds = time.perf_counter() - start
print(f"{n_nodes} x {n_nodes} VFNS solve: {solve_seconds:.2f} s")

In [ ]:
Main.b_plot_python = b_plot.tolist()
Main.mu_values_python = mu_values.tolist()
Main.eval(r"""
b_plot_vec = Float64.(b_plot_python)
mu_values_vec = Float64.(mu_values_python)
jq_plot = Matrix{Float64}(undef, length(mu_values_vec), length(b_plot_vec))
jg_plot = similar(jq_plot)

for i in eachindex(mu_values_vec), j in eachindex(b_plot_vec)
    value = solution(b_plot_vec[j], mu_values_vec[i])
    jq_plot[i, j] = value.jq
    jg_plot[i, j] = value.jg
end
""")

jq_plot = np.asarray(Main.jq_plot, dtype=float)
jg_plot = np.asarray(Main.jg_plot, dtype=float)
expected_shape = (mu_values.size, b_plot.size)
assert jq_plot.shape == expected_shape and jg_plot.shape == expected_shape
assert np.isfinite(jq_plot).all() and np.isfinite(jg_plot).all()
print(f"Jq range: [{jq_plot.min():.6g}, {jq_plot.max():.6g}]")
print(f"Jg range: [{jg_plot.min():.6g}, {jg_plot.max():.6g}]")

In [ ]:
# Collinear-EEC paper plotting convention used by the existing plot notebooks.
plt.rcParams.update({
    "text.usetex": shutil.which("latex") is not None,
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 15,
    "axes.linewidth": 1.0,
})

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)

for ax, mu, jq_values, jg_values in zip(axes.ravel(), mu_values, jq_plot, jg_plot):
    ax.plot(b_plot, jq_values, color="blue", linewidth=2.0, label=r"$J_q$")
    ax.plot(b_plot, jg_values, color="red", linewidth=2.0, linestyle="--", label=r"$J_g$")
    ax.text(
        0.95,
        0.93,
        rf"$\mu={mu:g}\ {{\rm GeV}}$",
        transform=ax.transAxes,
        ha="right",
        va="top",
    )
    ax.set_xscale("log")
    ax.set_xlim(0.01, 5.0)
    ax.minorticks_on()
    ax.tick_params(axis="both", direction="in", length=5, top=True, right=True)
    ax.tick_params(axis="both", which="minor", direction="in", length=2.5, top=True, right=True)

axes[0, 0].legend(frameon=False, loc="lower left")
fig.supxlabel(r"$b\ [{\rm GeV}^{-1}]$", fontsize=18)
fig.supylabel(r"$J_i(b,\mu)$", fontsize=18)
fig.subplots_adjust(left=0.08, right=0.99, bottom=0.09, top=0.99, wspace=0.0, hspace=0.0)

png_path = rg_dir / "jet_rg_vfns_2x3.png"
pdf_path = rg_dir / "jet_rg_vfns_2x3.pdf"
fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.show()

print(f"Saved {png_path}")
print(f"Saved {pdf_path}")